In [1]:
import os
import pandas as pd

# Dynamic path resolution: checks root path first, then parent directory
if os.path.exists(os.path.join("data", "raw", "smartphones_raw.csv")):
    RAW_DATA_PATH = os.path.join("data", "raw", "smartphones_raw.csv")
    PROCESSED_DATA_PATH = os.path.join("data", "processed", "processed_data.csv")
else:
    RAW_DATA_PATH = os.path.join("..", "data", "raw", "smartphones_raw.csv")
    PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "processed_data.csv")

# Load dataset
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

Raw dataset shape: (67986, 8)


,asin,name,rating,date,verified,title,body,helpfulVotes
0,B0000SX2UC,Janet,3,"October 11, 2005",False,"Def not best, but not worst",I had the Samsung A600 for awhile which is abs...,1.0
1,B0000SX2UC,Luke Wyatt,1,"January 7, 2004",False,Text Messaging Doesn't Work,Due to a software issue between Nokia and Spri...,17.0
2,B0000SX2UC,Brooke,5,"December 30, 2003",False,Love This Phone,"This is a great, reliable phone. I also purcha...",5.0
3,B0000SX2UC,amy m. teague,3,"March 18, 2004",False,"Love the Phone, BUT...!","I love the phone and all, because I really did...",1.0
4,B0000SX2UC,tristazbimmer,4,"August 28, 2005",False,"Great phone service and options, lousy case!",The phone has been great for every purpose it ...,1.0


In [2]:
# Display dataset column information
df_raw.info()

# Check total missing values per column
print("\nMissing values per column:")
print(df_raw.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 67986 entries, 0 to 67985
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   asin          67986 non-null  str    
 1   name          67983 non-null  str    
 2   rating        67986 non-null  int64  
 3   date          67986 non-null  str    
 4   verified      67986 non-null  bool   
 5   title         67957 non-null  str    
 6   body          67960 non-null  str    
 7   helpfulVotes  27215 non-null  float64
dtypes: bool(1), float64(1), int64(1), str(5)
memory usage: 3.7 MB

Missing values per column:
asin                0
name                3
rating              0
date                0
verified            0
title              29
body               26
helpfulVotes    40771
dtype: int64


In [3]:
import re

df = df_raw.copy()

# Standardize column names (lowercase with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Dynamically identify text and rating columns
text_col = next((col for col in ["body", "review", "review_text", "text"] if col in df.columns), None)
rating_col = next((col for col in ["rating", "ratings", "score"] if col in df.columns), None)

print(f"Using text column: '{text_col}' | rating column: '{rating_col}'")

# 1. Drop rows missing essential review text or ratings
df = df.dropna(subset=[text_col, rating_col])

# 2. Extract/Standardize Brand name
if "brand" not in df.columns and "title" in df.columns:
    df["brand"] = df["title"].astype(str).str.split().str[0].str.upper()
elif "brand" in df.columns:
    df["brand"] = df["brand"].astype(str).str.strip().str.upper()

# 3. Clean review text (remove excess spaces)
df["clean_review"] = df[text_col].astype(str).apply(lambda s: re.sub(r"\s+", " ", s).strip())

# 4. Remove duplicate reviews
df = df.drop_duplicates(subset=["clean_review"])

# 5. Add review word count feature & remove empty/short reviews
df["review_length"] = df["clean_review"].apply(lambda s: len(s.split()))
df = df[df["review_length"] >= 3]

print(f"Cleaned dataset shape: {df.shape}")
df[["brand", rating_col, "review_length", "clean_review"]].head()

Using text column: 'body' | rating column: 'rating'


Cleaned dataset shape: (57828, 11)


,brand,rating,review_length,clean_review
0,DEF,3,327,I had the Samsung A600 for awhile which is abs...
1,TEXT,1,129,Due to a software issue between Nokia and Spri...
2,LOVE,5,131,"This is a great, reliable phone. I also purcha..."
3,LOVE,3,107,"I love the phone and all, because I really did..."
4,GREAT,4,128,The phone has been great for every purpose it ...


In [4]:
# Ensure output directory exists
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Export cleaned data to processed folder
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Successfully saved clean dataset to: {PROCESSED_DATA_PATH}")

Successfully saved clean dataset to: ..\data\processed\processed_data.csv
